# Premotor neuronal activity

Run with `movement_bouts/` as the kernel's working directory.

`movement_bouts` is the reference table for all behavioral metadata. The neural
output has one `spike_counts` array per bout, indexed by the **same original bout
index**. Each array has shape `(neurons, 5 ms time bins)` and excludes movement onset.
Neuron IDs are stored once per session, indexed by `session_info.index` (`idx`).


# 1. Load the reference tables and define parameters


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

from utils import extract_all_premotor_bouts, prepare_pca_input

data_dir = Path("../../data").resolve()
results_dir = data_dir / "derived" / "movement_bouts"
premotor_dir = data_dir / "derived" / "premotor_activity"

movement_bouts = pd.read_pickle(results_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(results_dir / "session_info.pkl")
parameters = json.loads((results_dir / "parameters.json").read_text(encoding="utf-8"))

print(f"Loaded {len(movement_bouts)} movement bouts across {len(session_info)} session entries.")


In [ ]:
DT = parameters["DT"]
PREMOTOR_DURATION_S = 0.5  # Window: [-0.5, 0) seconds relative to movement onset.
N_PREMOTOR_BINS = round(PREMOTOR_DURATION_S / DT)
N_BINS = 5  # Averaging windows: 5 = 100 ms, 2 = 250 ms, 1 = 500 ms.


# 2. Extract and save the original-resolution counts

The helper loads each source collection once, then extracts session by session.
It skips bouts without a complete window and records their indices and reasons.
Behavioral metadata stays in `movement_bouts`; its indices are never reset.

To extract only one session, pass `session_info.loc[[idx]]` instead of `session_info`.


In [ ]:
neuronal_bouts, neuron_ids_by_session, excluded_bouts = extract_all_premotor_bouts(
    data_dir, movement_bouts, session_info,
    dt=DT, premotor_duration_s=PREMOTOR_DURATION_S,
)

print(f"Extracted {len(neuronal_bouts)} bouts; skipped {len(excluded_bouts)}.")
if len(excluded_bouts):
    display(excluded_bouts.head())


In [ ]:
import hashlib
from datetime import datetime, timezone

premotor_dir.mkdir(parents=True, exist_ok=True)
neuronal_bouts.to_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session.to_pickle(premotor_dir / "neuron_ids_by_session.pkl")
excluded_bouts.to_pickle(premotor_dir / "excluded_bouts.pkl")

# Fingerprints link bout indices to the exact reference tables used here.
extraction_parameters = {
    "DT": DT,
    "PREMOTOR_DURATION_S": PREMOTOR_DURATION_S,
    "N_PREMOTOR_BINS": N_PREMOTOR_BINS,
    "window_s": [-PREMOTOR_DURATION_S, 0],
    "onset_included": False,
    "units": "spike counts per original bin",
    "session_indices": [int(idx) for idx in neuron_ids_by_session.index],
    "source_directory": str(results_dir),
    "source_sha256": {
        name: hashlib.sha256((results_dir / name).read_bytes()).hexdigest()
        for name in ("movement_bouts.pkl", "session_info.pkl")
    },
    "bout_detection_parameters": parameters,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
}
(premotor_dir / "parameters.json").write_text(
    json.dumps(extraction_parameters, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved premotor tables to {premotor_dir}")


# 3. Prepare one session for PCA

If starting from the saved extraction, reload these two tables instead of
rerunning section 2. Run section 1 for the reference tables and parameters.

```python
neuronal_bouts = pd.read_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(premotor_dir / "neuron_ids_by_session.pkl")
```

`prepare_pca_input()` selects the session, averages counts into `N_BINS` equal
windows, converts to spikes/s, and creates one observation per bout/window.
`pca_input` contains only neural firing rates; `pca_labels` repeats all original
bout metadata and adds window labels. Both use `(bout_id, bin_start_s)` as their
index. Bout-level speed/direction labels are not recalculated per window.

Changing `idx` or `N_BINS` only requires rerunning this call and the cells below.
The helper keeps preparation in memory and does not fit PCA or save new files.


In [ ]:
idx = 0
pca_input, pca_labels = prepare_pca_input(
    neuronal_bouts, movement_bouts, neuron_ids_by_session,
    idx=idx, dt=DT, n_bins=N_BINS,
)

print(f"idx={idx}: PCA input {pca_input.shape} (bout-window observations, neurons)")
display(pca_input.head(N_BINS))
display(pca_labels.head(N_BINS))


# 4. Inspect one bout: all neurons and averaging windows


In [ ]:
import matplotlib.pyplot as plt

# Choose one retained bout by its position within the prepared session.
bout_number = 0
bout_id = pca_input.index.get_level_values("bout_id").unique()[bout_number]
bout_rates = pca_input.xs(bout_id, level="bout_id")
bout_labels = pca_labels.xs(bout_id, level="bout_id")
condition = bout_labels["condition"].iloc[0]
condition_name = {0: "Tracking", 1: "Playback"}.get(condition, str(condition))
cmap = {0: "Reds", 1: "Greys"}.get(condition, "viridis")
edges_ms = np.append(bout_rates.index.to_numpy(), bout_labels["bin_end_s"].iloc[-1]) * 1000
window_ms = edges_ms[1] - edges_ms[0]

fig, ax = plt.subplots(figsize=(11, max(4, len(bout_rates.columns) * 0.3)))
heatmap = ax.pcolormesh(
    edges_ms, np.arange(len(bout_rates.columns) + 1) - 0.5,
    bout_rates.to_numpy().T, cmap=cmap, vmin=0, shading="flat",
)
ax.set(
    yticks=np.arange(len(bout_rates.columns)), yticklabels=bout_rates.columns,
    xlabel="Time relative to movement onset (ms)", ylabel="Neuron",
    title=f"idx={idx} | bout ID {bout_id} | {condition_name} | {window_ms:g} ms windows",
    xlim=(edges_ms[0], edges_ms[-1]),
)
ax.invert_yaxis()
fig.colorbar(heatmap, ax=ax, label="Firing rate (spikes/s)")
fig.tight_layout()
plt.show()


# 5. Fit PCA within the selected session

Fit one common PCA using all prepared tracking and playback observations in this
session. Scikit-learn centers each neuron's firing rates before fitting; rates
are not standardized by their variance, and whitening is off. Fit all components
so we can inspect their explained variance before choosing how many to retain.

`pca_scores` retains the observation index. `pca_results` joins those scores to
all bout/window labels, for plotting or selecting either condition in the same
PCA space. The model and results stay in memory.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(svd_solver="full")
pca_scores = pd.DataFrame(
    pca.fit_transform(pca_input),
    index=pca_input.index,
    columns=[f"PC{i + 1}" for i in range(pca.n_components_)],
)
pca_results = pca_labels.join(pca_scores, validate="one_to_one")
explained_variance = pd.DataFrame(
    {
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_),
    },
    index=pca_scores.columns,
)

print(f"Fitted PCA for idx={idx}: {len(pca_scores)} observations, {pca.n_components_} components.")
display(explained_variance)
display(pca_results.head(N_BINS))
